# Step 1: Setup prerequisites

In [ ]:
import os
import sys
from pymongo import MongoClient

# utils에서 임포트하기 위해 부모 디렉토리를 경로에 추가합니다
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))
from utils import set_env

In [ ]:
# 본인의 MongoDB Atlas 클러스터를 사용하는 경우, 클러스터 연결 문자열을 여기에 입력하세요
MONGODB_URI = os.environ.get("MONGODB_URI")
# MongoDB Python 클라이언트 초기화
mongodb_client = MongoClient(MONGODB_URI)
# 서버 연결 확인
mongodb_client.admin.command("ping")

In [ ]:
# 워크숍 강사로부터 제공받은 LLM 프로바이더와 패스키를 설정하세요
# 참고: LLM_PROVIDER는 "aws", "microsoft", "google" 중 하나로 설정할 수 있습니다
LLM_PROVIDER = "aws"
PASSKEY = "replace-with-passkey"

In [ ]:
# AI 모델 프록시에서 API 키를 가져와 환경변수로 설정합니다 -- 변경하지 마세요
set_env([LLM_PROVIDER,"voyageai"], PASSKEY)

# Step 2: Import data into MongoDB

In [ ]:
import json

### **Do not change the values assigned to the variables below**

In [ ]:
# 데이터베이스 이름
DB_NAME = "mongodb_genai_devday_agents"
# 전체 문서가 있는 컬렉션 이름 -- 요약에 사용
FULL_COLLECTION_NAME = "mongodb_docs"
# 벡터 검색용 컬렉션 이름 -- Q&A에 사용
VS_COLLECTION_NAME = "mongodb_docs_embeddings"
# 벡터 검색 인덱스 이름
VS_INDEX_NAME = "vector_index"

In [ ]:
# `VS_COLLECTION_NAME` 컬렉션에 연결합니다
vs_collection = mongodb_client[DB_NAME][VS_COLLECTION_NAME]
# `FULL_COLLECTION_NAME` 컬렉션에 연결합니다
full_collection = mongodb_client[DB_NAME][FULL_COLLECTION_NAME]

In [ ]:
# Insert a dataset of MongoDB docs with embeddings into the `VS_COLLECTION_NAME` collection
with open(f"../data/{VS_COLLECTION_NAME}.json", "r") as data_file:
    json_data = data_file.read()

data = json.loads(json_data)

print(f"Deleting existing documents from the {VS_COLLECTION_NAME} collection.")
vs_collection.delete_many({})
vs_collection.insert_many(data)
print(
    f"{vs_collection.count_documents({})} documents ingested into the {VS_COLLECTION_NAME} collection."
)

In [ ]:
# Insert a dataset of MongoDB documentation pages into the `FULL_COLLECTION_NAME` collection
with open(f"../data/{FULL_COLLECTION_NAME}.json", "r") as data_file:
    json_data = data_file.read()

data = json.loads(json_data)

print(f"Deleting existing documents from the {FULL_COLLECTION_NAME} collection.")
full_collection.delete_many({})
full_collection.insert_many(data)
print(
    f"{full_collection.count_documents({})} documents ingested into the {FULL_COLLECTION_NAME} collection."
)

# Step 3: Create a vector search index

In [ ]:
from utils import create_index, check_index_ready

In [ ]:
# 벡터 인덱스 정의를 생성합니다:
# path: 임베딩 필드 경로
# numDimensions: 임베딩 차원 수 (사용하는 임베딩 모델에 따라 다름)
# similarity: 유사도 측정 방식. cosine, euclidean, dotProduct 중 하나
model = {
    "name": VS_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": 1024,
                "similarity": "cosine",
            }
        ]
    },
}

In [ ]:
# `utils` 모듈의 `create_index` 함수로 `vs_collection` 컬렉션에 위 정의로 벡터 검색 인덱스를 생성합니다
create_index(vs_collection, VS_INDEX_NAME, model)

In [ ]:
# `utils` 모듈의 `check_index_ready` 함수로 인덱스가 생성되고 READY 상태인지 확인한 후 진행합니다
check_index_ready(vs_collection, VS_INDEX_NAME)

# Step 4: Create agent tools


In [ ]:
from langchain.agents import tool
import voyageai
from typing import List

### Vector Search

In [ ]:
# Voyage AI 클라이언트 초기화
vo = voyageai.Client()

📚 https://docs.voyageai.com/docs/contextualized-chunk-embeddings#approach-2-contextualized-chunk-embeddings

In [ ]:
def get_embeddings(query: str) -> List[float]:
    """
    입력 쿼리에 대한 임베딩을 가져옵니다.

    Args:
        query (str): 쿼리 문자열

    Returns:
        List[float]: 쿼리 문자열의 임베딩 벡터
    """
    # Voyage AI API의 `contextualized_embed` 메서드로 사용자 쿼리를 임베딩합니다:
    # inputs: `query`를 리스트의 리스트로 감싼 것
    # model: `voyage-context-3`
    # input_type: "query"
    embds_obj = vo.contextualized_embed(inputs=[[query]], model="voyage-context-3", input_type="query")
    # 임베딩 객체에서 임베딩을 추출합니다
    embeddings = embds_obj.results[0].embeddings[0]
    return embeddings

📚 https://www.mongodb.com/docs/atlas/atlas-vector-search/vector-search-stage/#ann-examples (Refer to the "ANN Basic Example")

In [ ]:
# 벡터 검색으로 사용자 쿼리에 관련된 문서를 조회하는 도구를 정의합니다
@tool
def get_information_for_question_answering(user_query: str) -> str:
    """
    벡터 검색을 사용하여 사용자 쿼리에 답변하기 위한 정보를 검색합니다.

    Args:
    user_query (str): 사용자 쿼리 문자열.

    Returns:
    str: 문자열로 형식화된 검색 정보.
    """

    # 위에서 정의한 `get_embeddings` 함수로 `user_query`의 임베딩을 생성합니다
    query_embedding = get_embeddings(user_query)

    # $vectorSearch 스테이지와 $project 스테이지로 구성된 집계 파이프라인을 정의합니다
    # 후보 수는 150개로 설정하고 상위 5개만 반환합니다
    # $project 스테이지에서 `_id`는 제외하고 `body` 필드와 `vectorSearchScore`만 포함합니다
    # 참고: `index`, `queryVector`, `path` 필드에는 앞서 정의한 변수를 사용합니다
    pipeline = [
        {
            "$vectorSearch": {
                "index": VS_INDEX_NAME,
                "path": "embedding",
                "queryVector": query_embedding,
                "numCandidates": 150,
                "limit": 5,
            }
        },
        {
            "$project": {
                "_id": 0,
                "body": 1,
                "score": {"$meta": "vectorSearchScore"},
            }
        },
    ]

    # `vs_collection` 컬렉션에 집계 `pipeline`을 실행하고 결과를 `results`에 저장합니다
    results = vs_collection.aggregate(pipeline)
    # 결과를 하나의 문자열로 연결합니다
    context = "\n\n".join([doc.get("body") for doc in results])
    return context

### Get page content

📚 https://www.mongodb.com/docs/manual/reference/method/db.collection.findOne/#return-all-but-the-excluded-fields

In [ ]:
# 요약을 위한 문서 페이지 내용을 조회하는 도구를 정의합니다
@tool
def get_page_content_for_summarization(user_query: str) -> str:
    """
    제공된 제목을 기반으로 페이지 내용을 검색합니다.

    Args:
    user_query (str): 사용자 쿼리 문자열 (문서 페이지 제목).

    Returns:
    str: 페이지 내용.
    """
    # `title` 필드가 `user_query`와 같은 문서를 조회합니다
    query = {"title": user_query}
    # 검색된 문서에서 `body` 필드만 반환합니다
    # 참고: 포함할 필드는 1, 제외할 필드는 0으로 설정합니다. `_id`는 기본적으로 포함되므로 제외합니다
    projection = {"_id": 0, "body": 1}
    # `query`와 `projection`을 사용하여 `find_one` 메서드로
    # `full_collection` 컬렉션에서 `title`이 `user_query`와 같은 문서의 `body`를 가져옵니다
    document = full_collection.find_one(query, projection)
    if document:
        return document["body"]
    else:
        return "Document not found"

In [ ]:
# 도구 목록을 생성합니다
tools = [
    get_information_for_question_answering,
    get_page_content_for_summarization,
]

### Test out the tools


In [ ]:
# `get_information_for_question_answering` 도구를 "What are some best practices for data backups in MongoDB?" 쿼리로 테스트합니다
# 비어있지 않은 응답이 표시되어야 합니다
get_information_for_question_answering.invoke(
    "What are some best practices for data backups in MongoDB?"
)

In [ ]:
# `get_page_content_for_summarization` 도구를 "Create a MongoDB Deployment" 페이지 제목으로 테스트합니다
# 비어있지 않은 응답이 표시되어야 합니다
get_page_content_for_summarization.invoke("Create a MongoDB Deployment")

# Step 5: Instantiate the LLM

In [ ]:
from langchain_core.load import load
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from utils import get_llm

In [ ]:
# `utils` 모듈의 `get_llm` 함수로 LangChain LLM 객체를 가져옵니다
llm = get_llm(LLM_PROVIDER)

In [ ]:
# 에이전트를 위한 Chain-of-Thought(CoT) 프롬프트 템플릿을 생성합니다
# 도구 이름 플레이스홀더가 있는 시스템 프롬프트와 메시지(사용자 쿼리 및 어시스턴트 응답) 플레이스홀더를 포함합니다
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "You are a helpful AI assistant."
            " You are provided with tools to answer questions and summarize technical documentation related to MongoDB."
            " Think step-by-step and use these tools to get the information required to answer the user query."
            " Do not re-run tools unless absolutely necessary."
            " If you are not able to get enough information using the tools, reply with I DON'T KNOW."
            " You have access to the following tools: {tool_names}."
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

In [ ]:
# 프롬프트 템플릿에 도구 이름을 채웁니다
prompt = prompt.partial(tool_names=", ".join([tool.name for tool in tools]))

📚 https://docs.langchain.com/oss/python/langgraph/quickstart#1-define-tools-and-model

In [ ]:
# 위에서 초기화한 `llm`에 `tools`를 바인딩합니다
bind_tools = llm.bind_tools(tools)

📚 https://reference.langchain.com/python/langchain-core/runnables/base/Runnable/pipe (See Example)

In [ ]:
# `|` 연산자로 `prompt`와 도구가 바인딩된 llm을 체이닝합니다
llm_with_tools = prompt | bind_tools

In [ ]:
# LLM이 올바른 도구 호출을 하는지 테스트합니다
llm_with_tools.invoke(
    ["Give me a summary of the page titled Create a MongoDB Deployment."]
).tool_calls

In [ ]:
# LLM이 올바른 도구 호출을 하는지 테스트합니다
llm_with_tools.invoke(
    ["What are some best practices for data backups in MongoDB?"]
).tool_calls

# Step 6: Define graph state

In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

In [ ]:
# 그래프 상태를 정의합니다
# 채팅 메시지만 추적하지만 다른 속성도 추적할 수 있습니다
class GraphState(TypedDict):
    messages: Annotated[list, add_messages]

# Step 7: Define graph nodes

In [ ]:
from langchain_core.messages import ToolMessage
from typing import Dict
from pprint import pprint

In [ ]:
# 에이전트 노드를 정의합니다
def agent(state: GraphState) -> Dict[str, List]:
    """
    에이전트 노드

    Args:
        state (GraphState): 그래프 상태

    Returns:
        Dict[str, List]: 메시지 업데이트
    """
    # 그래프 `state`에서 메시지를 가져옵니다
    messages = state["messages"]
    # `invoke` 메서드를 사용하여 `messages`로 `llm_with_tools`를 호출합니다
    # 힌트: Step 6에서 `llm_with_tools`를 호출하는 방법을 참고하세요
    result = llm_with_tools.invoke(messages)
    # 그래프 상태의 `messages` 속성에 `result`를 씁니다
    return {"messages": [result]}

In [ ]:
# 도구 이름에서 도구 호출로의 맵을 생성합니다
tools_by_name = {tool.name: tool for tool in tools}
pprint(tools_by_name)

In [ ]:
# 도구 노드를 정의합니다
def tool_node(state: GraphState) -> Dict[str, List]:
    """
    도구 노드

    Args:
        state (GraphState): 그래프 상태

    Returns:
        Dict[str, List]: 메시지 업데이트
    """
    result = []
    # 메시지에서 도구 호출 목록을 가져옵니다
    tool_calls = state["messages"][-1].tool_calls
    # tool_call의 구조는 다음과 같습니다:
    # {
    #     "name": "get_information_for_question_answering",
    #     "args": {"user_query": "What are Atlas Triggers"},
    #     "id": "call_H5TttXb423JfoulF1qVfPN3m",
    #     "type": "tool_call",
    # }
    # `tool_calls`를 순회합니다
    for tool_call in tool_calls:
        # `tool_call`의 `name` 속성으로 `tools_by_name`에서 도구를 가져옵니다
        tool = tools_by_name[tool_call["name"]]
        # `tool_call`의 `args` 속성으로 `tool`을 호출합니다
        # 힌트: `tool_call`에서 속성을 추출하는 방법은 위 줄을 참고하세요
        observation = tool.invoke(tool_call["args"])
        # 도구 실행 결과를 ToolMessage로 `result` 목록에 추가합니다
        # 메시지의 `content`는 도구 호출 결과인 `observation`입니다
        # `tool_call_id`는 `tool_call`에서 가져올 수 있습니다
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    # 그래프 상태의 `messages` 속성에 `result`를 씁니다
    return {"messages": result}

# Step 8: Define conditional edges

In [ ]:
from langgraph.graph import END

In [ ]:
# 조건부 라우팅 함수를 정의합니다
def route_tools(state: GraphState):
    """
    마지막 메시지에 도구 호출이 있으면 도구 노드로,
    그렇지 않으면 종료로 라우팅하는 조건부 엣지 함수입니다.
    """
    # 그래프 상태에서 메시지를 가져옵니다
    messages = state.get("messages", [])
    if len(messages) > 0:
        # 메시지에서 마지막 AI 메시지를 가져옵니다
        ai_message = messages[-1]
    else:
        raise ValueError(f"No messages found in input state to tool_edge: {state}")
    # 마지막 메시지에 도구 호출이 있는지 확인합니다
    if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0:
        # 있으면 "tools"를 반환합니다
        return "tools"
    # 없으면 END를 반환합니다
    return END

# Step 9: Build the graph

In [ ]:
from langgraph.graph import StateGraph, START
from IPython.display import Image, display

In [ ]:
# Instantiate the graph
graph = StateGraph(GraphState)

📚 https://docs.langchain.com/oss/python/langgraph/graph-api#nodes

In [ ]:
# `add_node` 함수로 `graph`에 노드를 추가합니다
# `agent` 함수를 실행하는 `agent` 노드를 추가합니다
graph.add_node("agent", agent)
# `tool_node` 함수를 실행하는 `tools` 노드를 추가합니다
graph.add_node("tools", tool_node)

📚 https://docs.langchain.com/oss/python/langgraph/graph-api#normal-edges

In [ ]:
# `add_edge` 메서드로 `graph`에 고정 엣지를 추가합니다
# START 노드에서 `agent` 노드로의 엣지를 추가합니다
graph.add_edge(START, "agent")
# `tools` 노드에서 `agent` 노드로의 엣지를 추가합니다
graph.add_edge("tools", "agent")

📚 https://docs.langchain.com/oss/python/langgraph/graph-api#conditional-edges

In [ ]:
# `add_conditional_edges` 메서드로 `route_tools` 함수의 출력에 따라
# `agent` 노드에서 `tools` 노드로의 조건부 엣지를 추가합니다
graph.add_conditional_edges(
    "agent",
    route_tools,
    {"tools": "tools", END: END},
)

In [ ]:
# `graph`를 컴파일합니다
app = graph.compile()

In [ ]:
# 그래프를 시각화합니다
app

# Step 10: Execute the graph

In [ ]:
def execute_graph(user_input: str) -> None:
    """
    그래프에서 출력을 스트리밍합니다

    Args:
        user_input (str): 사용자 쿼리 문자열
    """
    # 그래프의 각 단계에서 출력을 스트리밍합니다
    for step in app.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        # 각 단계 후 상태의 전체 값을 스트리밍합니다
        stream_mode="values",
    ):
        # 단계의 최신 메시지를 출력합니다
        step["messages"][-1].pretty_print()

In [ ]:
# 엔드투엔드 흐름을 확인하기 위해 그래프 실행을 테스트합니다
execute_graph("What are some best practices for data backups in MongoDB?")

In [ ]:
# 엔드투엔드 흐름을 확인하기 위해 그래프 실행을 테스트합니다
execute_graph("Give me a summary of the page titled Create a MongoDB Deployment")

# Step 11: Add short-term memory to the agent

In [ ]:
from langgraph.checkpoint.mongodb import MongoDBSaver

In [ ]:
# MongoDB 체크포인터를 초기화합니다
checkpointer = MongoDBSaver(mongodb_client)

In [ ]:
# 체크포인터를 사용하여 그래프를 인스턴스화합니다
app = graph.compile(checkpointer=checkpointer)

📚 https://docs.langchain.com/oss/python/langgraph/persistence#threads

In [ ]:
def execute_graph(thread_id: str, user_input: str) -> None:
    """
    그래프에서 출력을 스트리밍합니다

    Args:
        thread_id (str): 체크포인터의 스레드 ID
        user_input (str): 사용자 쿼리 문자열
    """
    # 스레드 ID `thread_id`에 대한 런타임 설정을 생성합니다
    config = {"configurable": {"thread_id": thread_id}}
    # 그래프의 각 단계에서 출력을 스트리밍합니다
    for step in app.stream(
        {"messages": [{"role": "user", "content": user_input}]},
        # 추가 파라미터로 config를 전달합니다
        config,
        stream_mode="values",
    ):
        # 단계의 최신 메시지를 출력합니다
        step["messages"][-1].pretty_print()

In [ ]:
# 스레드 ID로 그래프 실행을 테스트합니다
execute_graph(
    "1",
    "What are some best practices for data backups in MongoDB?",
)

In [ ]:
# 메시지 기록이 작동하는지 확인하기 위한 후속 질문입니다
execute_graph(
    "1",
    "What did I just ask you?",
)

# 🦹‍♀️ Step 12: Add long-term memory to the agent

📚 https://docs.langchain.com/oss/python/langgraph/add-memory#use-semantic-search

In [ ]:
from langgraph.store.mongodb import MongoDBStore, create_vector_index_config
from langchain_voyageai import VoyageAIEmbeddings
import uuid

In [ ]:
# 장기 메모리 저장을 위한 MongoDB 컬렉션을 초기화합니다
memory_collection = mongodb_client[DB_NAME]["memories"]

In [ ]:
# 벡터 검색으로 메모리를 검색하기 위해 Voyage AI 임베딩을 사용하는 MongoDB 장기 메모리 스토어를 초기화합니다
mongodb_store = MongoDBStore(
    collection=memory_collection,
    index_config=create_vector_index_config(
        embed = VoyageAIEmbeddings(model="voyage-4-large"),
        dims = 1024,
    )
)

In [ ]:
# MongoDB 스토어에 메모리를 저장하는 도구를 생성합니다
@tool
def save_memory(memory: str) -> str:
    """
    향후 대화를 위해 사용자에 대한 중요한 사실과 선호도를 저장합니다.

    Args:
    memory: 기억할 정보
    """
    mongodb_store.put(
        # 메모리 항목의 네임스페이스. 다른 유형의 메모리를 저장하기 위한 하위 네임스페이스도 사용 가능 (예: ("user_1", "preferences"))
        # 빈 값을 포함하더라도 튜플이어야 합니다
        ("user_1",),
        # 고유 메모리 ID
        key=str(uuid.uuid4()),
        # 메모리 내용. 딕셔너리여야 합니다
        value={"text": memory}
    )
    return f"Memory saved: {memory}"

In [ ]:
# `save_memory` 도구를 포함하도록 도구 목록을 업데이트합니다
tools = [
      get_information_for_question_answering,
      get_page_content_for_summarization,
      save_memory,
  ]
tools_by_name = {tool.name: tool for tool in tools}
# LLM에 도구를 바인딩합니다
bind_tools = llm.bind_tools(tools)

In [ ]:
# 응답 시 관련 장기 메모리를 검색하도록 에이전트 노드를 업데이트합니다
def agent(state: GraphState) -> Dict[str, List]:
    """
    에이전트 노드

    Args:
        state (GraphState): 그래프 상태

    Returns:
        Dict[str, List]: 메시지 업데이트
    """
    # 그래프 `state`에서 `messages`를 가져옵니다
    messages = state["messages"]
    # 사용자의 마지막 메시지로 관련 장기 메모리를 검색합니다
    memories = mongodb_store.search(  
        ("user_1",), query=messages[-1].content, limit=10
    )
    # 검색된 메모리를 문자열로 형식화합니다
    memories = "\n".join(m.value["text"] for m in memories)
    memories = memories if memories else "No memories stored yet for this user."
    # 메모리를 포함하는 시스템 프롬프트를 작성합니다
    system_prompt = (
        "You are a helpful AI assistant."
        "You are provided with tools to answer questions and summarize technical documentation related to MongoDB."
        "Think step-by-step and use these tools to get the information required to answer the user query."
        "Do not re-run tools unless absolutely necessary."
        "If you are not able to get enough information using the tools, reply with I DON'T KNOW."
        f" You have access to the following tools: {', '.join([t.name for t in tools])}."
        "If the user shares any preferences, extract them and save them as memories using the save_memory tool."
        "User past user preferences to personalize future conversations."
        "Past user memories:"
        f"{memories}"
    )
    # 시스템 프롬프트와 메시지를 입력으로 도구가 바인딩된 LLM을 호출합니다
    result = bind_tools.invoke([
        {"role": "system", "content": system_prompt},
        *messages,
    ])
    # 그래프 상태의 `messages` 속성에 `result`를 씁니다
    return {"messages": [result]}

In [ ]:
# 에이전트 그래프를 재구성합니다
graph = StateGraph(GraphState)
graph.add_node("agent", agent)
graph.add_node("tools", tool_node)
graph.add_edge(START, "agent")
graph.add_edge("tools", "agent")
graph.add_conditional_edges(
    "agent",
    route_tools,
    {"tools": "tools", END: END},
)

In [ ]:
# 단기 메모리를 위한 MongoDB 체크포인터와 장기 메모리를 위한 MongoDB 메모리 스토어로 그래프를 컴파일합니다
app = graph.compile(checkpointer=checkpointer, store=mongodb_store)

In [ ]:
# user_1 사용자의 스레드 1에서 메모리 생성을 테스트합니다
execute_graph(
    "1",
    "Remember that I prefer detailed explanations with code examples when learning about MongoDB features."
)

In [ ]:
# 동일한 스레드에서 후속 질문 -- 단기 메모리가 작동하는지 확인합니다
execute_graph(
    "1",
    "What do you know about my learning preferences?"
)

In [ ]:
# 새 스레드에서 후속 질문 -- 세션 간 장기 메모리가 작동하는지 확인합니다
execute_graph(
    "2",
    "Hi! Do you remember anything about how I like to learn about MongoDB?"
)

In [ ]:
# Atlas Search에 대해 질문합니다 -- 에이전트가 저장된 선호도에 따라 상세한 설명을 제공해야 합니다
execute_graph(
    "2",
    "What is Atlas Search in MongoDB?"
)